# 03 — Training Dynamics: How Did Each Model Actually Learn?

**Goal of this notebook**: `notebooks/06_model_comparison.ipynb` shows final test
accuracy. This notebook looks at the *process* — how loss and accuracy evolved epoch
by epoch, how large the train/val gap got (overfitting signal), and what visibly
happens to DenseNet's validation curve at the exact epoch where Phase 2 (unfreezing)
begins.

**Intent**: reads the CSV training logs that `src/train.py`'s `CSVLogger` callback
already wrote during training — no retraining needed, so this stays fast to re-run
and iterate on.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

CHECKPOINT_DIR = "../models/saved_models"

## Load training histories

In [ ]:
custom_cnn_history = pd.read_csv(f"{CHECKPOINT_DIR}/custom_cnn_baseline_history.csv")
densenet_phase1_history = pd.read_csv(f"{CHECKPOINT_DIR}/densenet121_phase1_frozen_history.csv")
densenet_phase2_history = pd.read_csv(f"{CHECKPOINT_DIR}/densenet121_phase2_finetuned_history.csv")

# Stitch phase 1 + phase 2 into one continuous timeline for DenseNet
densenet_full_history = pd.concat(
    [densenet_phase1_history, densenet_phase2_history], ignore_index=True
)
phase_boundary_epoch = len(densenet_phase1_history)

## Train vs. validation accuracy — overfitting gap

**Intent**: a large, growing gap between train and val accuracy is the classic
overfitting signature. Given the custom CNN has much less effective training data to
work with (no pretrained knowledge to fall back on), expect its gap to be larger and
appear earlier than DenseNet's.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(custom_cnn_history["accuracy"], label="train")
axes[0].plot(custom_cnn_history["val_accuracy"], label="val")
axes[0].set_title("Custom CNN — accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(densenet_full_history["accuracy"], label="train")
axes[1].plot(densenet_full_history["val_accuracy"], label="val")
axes[1].axvline(phase_boundary_epoch, color="gray", linestyle="--", label="Phase 1 → 2 (unfreeze)")
axes[1].set_title("DenseNet121 (transfer) — accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## Quantify the overfitting gap directly

**Intent**: eyeballing a chart is useful, but a single number — final train accuracy
minus final val accuracy — makes the comparison concrete and citable in a report.

In [ ]:
def final_gap(history):
    return history["accuracy"].iloc[-1] - history["val_accuracy"].iloc[-1]


print(f"Custom CNN final train/val accuracy gap: {final_gap(custom_cnn_history):.4f}")
print(f"DenseNet121 final train/val accuracy gap: {final_gap(densenet_full_history):.4f}")

## The unfreezing effect, close up

**Intent**: zoom into just the epochs around the phase boundary. A visible dip or jump
in val accuracy right at unfreezing is expected — the model's later layers are
adapting to MRI-specific features rather than pure ImageNet features, which can
temporarily disrupt performance before it improves further.

In [ ]:
window = 3
start = max(0, phase_boundary_epoch - window)
end = min(len(densenet_full_history), phase_boundary_epoch + window)

zoomed = densenet_full_history.iloc[start:end]
plt.figure(figsize=(8, 5))
plt.plot(zoomed.index, zoomed["val_accuracy"], marker="o")
plt.axvline(phase_boundary_epoch, color="gray", linestyle="--", label="Unfreeze point")
plt.title("DenseNet121 — validation accuracy around the unfreezing point")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.show()

## Loss curves — a second view of the same story

**Intent**: accuracy can plateau while loss keeps improving (the model becomes more
*confident*, not just more often correct) — loss curves sometimes reveal training
dynamics accuracy alone hides.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(custom_cnn_history["loss"], label="train")
axes[0].plot(custom_cnn_history["val_loss"], label="val")
axes[0].set_title("Custom CNN — loss")
axes[0].legend()

axes[1].plot(densenet_full_history["loss"], label="train")
axes[1].plot(densenet_full_history["val_loss"], label="val")
axes[1].axvline(phase_boundary_epoch, color="gray", linestyle="--")
axes[1].set_title("DenseNet121 — loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusion

Fill in after running: report the actual overfitting-gap numbers, describe what
happened at the unfreezing point (did val accuracy dip then recover, or improve
smoothly?), and note which model's training looked more stable overall. This is good
supporting material for explaining *why* you chose your final model, not just *that*
it scored higher.

Next notebook: **04_layer_freezing_ablation.ipynb** — a systematic experiment on how
many layers to unfreeze, rather than just the one setting (30) used above.